# PlantVillage (Kaggle) -> Prepare -> Fine-tune -> Push to Hugging Face (Colab)

This notebook automates: download PlantVillage dataset from Kaggle, prepare ImageFolder splits, fine-tune a ViT/ResNet model using the included training script, and push the trained model to the Hugging Face Hub.

Notes: set the Colab runtime to **GPU** (Runtime > Change runtime type > GPU). You will need a Kaggle API token (or username/key) and a Hugging Face token (HF token) to push the model.


## 1) Install dependencies

In [ ]:
# Install required Python packages (may take several minutes)
!pip install -q transformers accelerate evaluate huggingface_hub torch torchvision timm kaggle
# Verify versions (optional)
python -c "import transformers, torch; print('transformers', transformers.__version__, 'torch', torch.__version__)"

## 2) (Optional) Mount Google Drive to persist downloads / outputs

In [ ]:
# Uncomment to mount Google Drive and set a base directory for datasets and outputs
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/plant_finetune'
# os.makedirs(BASE_DIR, exist_ok=True)
BASE_DIR = '/content'


## 3) Provide Kaggle credentials
You can upload your kaggle.json (preferred) or set KAGGLE_USERNAME/KAGGLE_KEY as environment variables in Colab.


In [ ]:
# Option A: upload kaggle.json (recommended for Colab)
from google.colab import files
print('Please upload your kaggle.json (from https://www.kaggle.com/ -> Account -> API).')
uploaded = files.upload()
if uploaded:
  # move kaggle.json to ~/.kaggle/kaggle.json
  import os, io
  os.makedirs('/root/.kaggle', exist_ok=True)
  for fn in uploaded:
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
      f.write(uploaded[fn])
  os.chmod('/root/.kaggle/kaggle.json', 0o600)
  print('kaggle.json saved to /root/.kaggle/kaggle.json')


## 4) Download PlantVillage dataset from Kaggle and prepare splits
The default dataset slug used here is `emmarex/plantdisease` (PlantVillage on Kaggle).


In [ ]:
# Download and prepare dataset (this uses the download_plantvillage_kaggle.py script included in the repo)
DATASET_SLUG = 'emmarex/plantdisease'
RAW_OUT = f'{BASE_DIR}/plantvillage_raw'
HF_PREPARE_OUT = f'{BASE_DIR}/hf_dataset'
!kaggle kernels pull imtkaggleteam/plant-diseases-detection-pytorch -p /content/kaggle_kernel --unzip || true
!ls -la /content/kaggle_kernel || true
!python scripts/hf_finetune/download_plantvillage_kaggle.py --dataset {DATASET_SLUG} --out_dir {RAW_OUT} --prepare_out {HF_PREPARE_OUT} || true
print('If the script failed, inspect the folder:', RAW_OUT)


## 5) (Optional) Inspect prepared dataset
You should see `train/`, `validation/`, `test/` folders under the prepared output.


In [ ]:
!ls -la {HF_PREPARE_OUT} || true
!find {HF_PREPARE_OUT} -maxdepth 2 -type d -print | sed -n '1,200p' || true


## 6) Login to Hugging Face (to push the trained model)
You can enter your Hugging Face token securely below. This will call `huggingface_hub.login()` so trainer.push_to_hub() can authenticate.


In [ ]:
from getpass import getpass
from huggingface_hub import login
hf_token = getpass('Enter your Hugging Face token (will not be shown):')
if hf_token:
  login(token=hf_token)
  import os
  os.environ['HF_TOKEN'] = hf_token
  print('Logged in to Hugging Face')
else:
  print('No token provided — you can still train locally but push_to_hub will fail')


## 7) Start training
Set `HUB_MODEL_ID` to your desired model ID on Hugging Face (e.g. `your-username/plant-vit`).


In [ ]:
# Adjust these parameters if needed
HUB_MODEL_ID = 'your-username/plant-vit'  # <-- change this to your HF repo id
MODEL_NAME = 'google/vit-base-patch16-224'
DATA_DIR = HF_PREPARE_OUT
OUTPUT_DIR = f'{BASE_DIR}/outputs/plant-vit'
!python scripts/hf_finetune/train.py --dataset_path {DATA_DIR} --model_name_or_path {MODEL_NAME} --output_dir {OUTPUT_DIR} --per_device_train_batch_size 16 --num_train_epochs 6 --push_to_hub --hub_model_id {HUB_MODEL_ID} || true


## 8) After training
- If `--push_to_hub` succeeded your model will be available at the Hub model page you specified.
- Set `HUGGINGFACE_MODEL` environment variable on your server to this model id and redeploy the app so server-side inference uses it.
  Example: `HUGGINGFACE_MODEL=your-username/plant-vit`
